In [1]:
import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

c:\Users\Abc\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


In [2]:
sentences = [
    "An old railway station stood silent after the last train left decades ago.",
    "The abandoned village was slowly being covered by trees and vines.",
    "Broken windows reflected the sunset inside the empty factory.",
    "A forgotten amusement park still had rusted rides near the entrance.",
    "The deserted hospital contained long hallways filled with dust.",
    "An entire mining town was abandoned when the resources ran out.",
    "Nature slowly reclaimed the buildings that people had left behind.",
    "A cracked sign still displayed the name of the forgotten town.",
    "Old photographs were discovered inside one of the abandoned houses.",
    "The empty school still had writing on several classroom blackboards.",
    "A lighthouse on a remote island had not been used for many years.",
    "The abandoned hotel once welcomed hundreds of travelers every season.",
    "Wild animals now moved freely through the streets of the deserted city.",
    "An old cinema still contained rows of damaged red seats.",
    "The underground tunnel had been sealed and forgotten for nearly a century.",
    "Several abandoned buildings were preserved because of their historical value.",
    "Explorers often visit forgotten places to document their unusual history.",
    "A flooded village became visible again when the water level dropped.",
    "The remains of an ancient settlement were discovered beneath modern buildings.",
    "Every abandoned place carries clues about the people who once lived there."
]

In [3]:
for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")

1. An old railway station stood silent after the last train left decades ago.
2. The abandoned village was slowly being covered by trees and vines.
3. Broken windows reflected the sunset inside the empty factory.
4. A forgotten amusement park still had rusted rides near the entrance.
5. The deserted hospital contained long hallways filled with dust.
6. An entire mining town was abandoned when the resources ran out.
7. Nature slowly reclaimed the buildings that people had left behind.
8. A cracked sign still displayed the name of the forgotten town.
9. Old photographs were discovered inside one of the abandoned houses.
10. The empty school still had writing on several classroom blackboards.
11. A lighthouse on a remote island had not been used for many years.
12. The abandoned hotel once welcomed hundreds of travelers every season.
13. Wild animals now moved freely through the streets of the deserted city.
14. An old cinema still contained rows of damaged red seats.
15. The underground 

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1487.52it/s]


Embedding model loaded successfully!


# semantic search works with vectors.

Sentence
   ->
SentenceTransformer
   ->
Embedding
   ->
Vector

In [5]:
embeddings = model.encode(sentences)
print("Embeddings created successfully !")
print("embedding shape :", embeddings.shape)

Embeddings created successfully !
embedding shape : (20, 384)


In [6]:
embeddings

array([[ 0.02722509, -0.00434437, -0.00621918, ...,  0.00017721,
        -0.01238046,  0.04367637],
       [ 0.06556621,  0.07787067, -0.05372408, ..., -0.03469347,
         0.02265951, -0.00524195],
       [ 0.01160294,  0.04525131,  0.11614391, ...,  0.00483863,
        -0.02970346,  0.09960759],
       ...,
       [-0.02971408, -0.00112173,  0.038473  , ..., -0.01217922,
        -0.07494494,  0.03296789],
       [-0.01999498,  0.12736379,  0.05488881, ..., -0.01785468,
         0.0388968 ,  0.00254761],
       [ 0.08993959,  0.0117441 ,  0.05166658, ...,  0.05010105,
        -0.02498498,  0.03520455]], shape=(20, 384), dtype=float32)

In [7]:
print("Sentence:")
print(sentences[0])

print("\nFirst 10 values of its embedding:")
print(embeddings[0][:10])

Sentence:
An old railway station stood silent after the last train left decades ago.

First 10 values of its embedding:
[ 0.02722509 -0.00434437 -0.00621918  0.10221344  0.02636717  0.09009898
 -0.05912298 -0.00542074 -0.06965112 -0.12568195]


In [8]:
np.save("sentence_embeddings.npy", embeddings)

print("Embeddings saved to sentence_embeddings.npy")


Embeddings saved to sentence_embeddings.npy


# embedding completed noe similarity 

how computer understand what people say ?

we convert the query into a embedding too...

query 
   ->
embedding model
     ->
query vector

### Cosine similarity

we will use cosine similarity to decide which vectors are closest

In [9]:
def semantic_search(query, top_k=3):

    # Step 1: Convert the user's query into an embedding
    query_embedding = model.encode([query])

    # Step 2: Compare query embedding with all sentence embeddings
    similarity_scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Step 3: Find indices of highest similarity scores
    top_indices = similarity_scores.argsort()[::-1][:top_k]

    # Step 4: Store the top results
    results = []

    for index in top_indices:
        results.append({
            "sentence": sentences[index],
            "similarity": float(similarity_scores[index])
        })

    return results

In [10]:
query = "How do people will know the town is abondend ?"

results = semantic_search(query)

print("Query:")
print(query)

print("\nTop 3 Semantic Search Results:")

for rank, result in enumerate(results, start=1):
    print(f"\n{rank}. {result['sentence']}")
    print(f"   Similarity Score: {result['similarity']:.4f}")

Query:
How do people will know the town is abondend ?

Top 3 Semantic Search Results:

1. Every abandoned place carries clues about the people who once lived there.
   Similarity Score: 0.5135

2. A cracked sign still displayed the name of the forgotten town.
   Similarity Score: 0.4593

3. Wild animals now moved freely through the streets of the deserted city.
   Similarity Score: 0.3013


In [11]:
# testing for keyword search 

def clean_words(text):

    # Convert text to lowercase
    text = text.lower()

    # Extract words and remove punctuation
    words = re.findall(r"\b\w+\b", text)

    # Some common words that are not useful for keyword matching
    stop_words = {
        "a", "an", "the", "is", "are", "was", "were",
        "to", "of", "and", "in", "on", "for", "with",
        "how", "what", "do", "does", "can", "from"
    }

    return {
        word
        for word in words
        if word not in stop_words
    }


def keyword_search(query, top_k=3):

    query_words = clean_words(query)

    results = []

    for sentence in sentences:

        sentence_words = clean_words(sentence)

        # Exact words appearing in both query and sentence
        matching_words = query_words.intersection(sentence_words)

        score = len(matching_words)

        results.append({
            "sentence": sentence,
            "score": score,
            "matching_words": list(matching_words)
        })

    # Sort from highest score to lowest
    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_k]

In [12]:
query = "How do people will know the town is abondend ?"

semantic_results = semantic_search(query)
keyword_results = keyword_search(query)

print("=" * 70)
print("QUERY")
print("=" * 70)
print(query)


print("\n" + "=" * 70)
print("SEMANTIC SEARCH - TOP 3")
print("=" * 70)

for rank, result in enumerate(semantic_results, start=1):
    print(f"\n{rank}. {result['sentence']}")
    print(f"Similarity: {result['similarity']:.4f}")


print("\n" + "=" * 70)
print("KEYWORD SEARCH - TOP 3")
print("=" * 70)

for rank, result in enumerate(keyword_results, start=1):
    print(f"\n{rank}. {result['sentence']}")
    print(f"Keyword Score: {result['score']}")
    print(f"Matching Words: {result['matching_words']}")

QUERY
How do people will know the town is abondend ?

SEMANTIC SEARCH - TOP 3

1. Every abandoned place carries clues about the people who once lived there.
Similarity: 0.5135

2. A cracked sign still displayed the name of the forgotten town.
Similarity: 0.4593

3. Wild animals now moved freely through the streets of the deserted city.
Similarity: 0.3013

KEYWORD SEARCH - TOP 3

1. An entire mining town was abandoned when the resources ran out.
Keyword Score: 1
Matching Words: ['town']

2. Nature slowly reclaimed the buildings that people had left behind.
Keyword Score: 1
Matching Words: ['people']

3. A cracked sign still displayed the name of the forgotten town.
Keyword Score: 1
Matching Words: ['town']


In [13]:
query = "Why water level dropped in village ?"

results = semantic_search(query)

print("Query:", query)

print("\nTop 3 Results:")

for rank, result in enumerate(results, start=1):
    print(f"\n{rank}. {result['sentence']}")
    print(f"Similarity: {result['similarity']:.4f}")

Query: Why water level dropped in village ?

Top 3 Results:

1. A flooded village became visible again when the water level dropped.
Similarity: 0.7420

2. The abandoned village was slowly being covered by trees and vines.
Similarity: 0.3134

3. Nature slowly reclaimed the buildings that people had left behind.
Similarity: 0.2901


In [ ]:
query = "How does an app suggest songs that I may like?"
# also works with different questions 

results = semantic_search(query)

print("Query:", query)

print("\nTop 3 Results:")

for rank, result in enumerate(results, start=1):
    print(f"\n{rank}. {result['sentence']}")
    print(f"Similarity: {result['similarity']:.4f}")

Query: How does an app suggest songs that I may like?

Top 3 Results:

1. Explorers often visit forgotten places to document their unusual history.
Similarity: 0.1179

2. A forgotten amusement park still had rusted rides near the entrance.
Similarity: 0.0831

3. Every abandoned place carries clues about the people who once lived there.
Similarity: 0.0744
